# 02 - Real token savings (OpenAI)

**Requires `OPENAI_API_KEY`.** Estimated cost to run: about **$0.01-0.05** (gpt-4o-mini extractor + two gpt-4o narrative generations + one judge call).

What this notebook proves:
1. Real token counts before and after compression on a 4 KB document.
2. Real USD cost difference at current OpenAI prices.
3. Quality scored by an independent LLM judge so the savings are not free.

Honest expectation: on this size of document you will see **40-60% token reduction** in the downstream prompt and **10-30% net cost reduction** after paying for the extractor call. The bigger the source, the bigger the cost win.


## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # picks up OPENAI_API_KEY from .env if present

assert os.getenv("OPENAI_API_KEY"), "set OPENAI_API_KEY (env or .env) before running"

from narrato import Compressor
from narrato.benchmark import run_benchmark


## The source document

A 4 KB English transcript-style document with realistic redundancy. Replace with your own — the bigger the input, the bigger the savings.


In [ ]:
SOURCE = '''\
Customer support call transcript, March 14, 2026.
Agent: Maria.  Customer: ID 481-2207.

Maria: Hello, this is Maria with Acme Cloud support. How can I help you today?
Customer: Hi Maria. I'm calling because my team can't access the production database since
this morning. We've been getting connection timeouts since about 8 AM Pacific.

Maria: I'm sorry to hear that. Can you confirm your account ID? ... Got it, 481-2207.
Let me pull up your environment. ... I can see you're on the prod-west cluster, db-7f.
Customer: Right.

Maria: I see in our incident log that there was a maintenance window scheduled for prod-west
between 7 and 7:45 AM Pacific. It looks like that window ran long because of a failed failover.
We're showing the cluster fully back at 9:12 AM, which was about twenty minutes ago.
Customer: We're still seeing timeouts on our end.

Maria: Okay. When you say timeouts, are you seeing them from your application servers or from
developer laptops, or both?
Customer: From our app servers, mostly. The developers tested manually and it seems fine for
them.

Maria: Got it. That pattern usually means the connection pool on your app servers is still
holding stale connections from before the failover. They'll keep timing out until the pool
recycles. Two options: you can wait for the pool TTL to expire, which is usually 15-30 minutes,
or you can restart the app servers and force fresh connections.
Customer: We can do a rolling restart. That's fine.

Maria: Great. I'd also recommend dropping your connection pool max-age to 5 minutes
temporarily, so if we have another failover today the recovery is faster.
Customer: Yeah, makes sense. We have it at 60 minutes right now.

Maria: One more thing. Our incident report will go out by end of day with full root cause.
I'll put your account on the priority distribution list so you get it directly.
Customer: Thanks, that helps. Anything else we should do?

Maria: No, you're set. Rolling restart, drop the pool TTL temporarily, and watch for the
incident report. If you see timeouts again after the restart, call back and reference incident
INC-19842.
Customer: INC-19842, got it. Thank you Maria.
Maria: You're welcome. Have a good day.
'''

print(f"source length: {len(SOURCE)} chars")


## Run the benchmark

`run_benchmark` does the following, end-to-end:
1. Generates a *baseline* narrative from the **full** source with gpt-4o.
2. Compresses the source with `narratoflow` (gpt-4o-mini extractor).
3. Generates a *compressed* narrative from the dense payload with gpt-4o.
4. Asks gpt-4o-mini to score the compressed narrative against the baseline.

Outputs token counts, USD cost, and a 1-10 quality score.


In [ ]:
compressor = Compressor.from_profile(
    "interview-en",
    provider="openai",
    extractor_model="gpt-4o-mini",
    target_model="gpt-4o",
)

report = run_benchmark(
    SOURCE,
    instruction="Write a concise (150-word) summary suitable for an incident postmortem ticket.",
    compressor=compressor,
    target_model="gpt-4o",
    judge_model="gpt-4o-mini",
)

print(report.to_json())


## Read the result

Key fields to look at:
- `tokens_source` vs `tokens_compressed` — raw reduction in the downstream prompt
- `cost_baseline` vs `cost_compressed` — net dollar amount after paying for the extractor
- `cost_savings_pct` — the headline number
- `quality_score` — independent judge on a 1-10 scale
- `quality_rationale` — *why* the judge gave that score, including what it noticed missing


In [ ]:
print(f"tokens_source     : {report.tokens_source:>8}")
print(f"tokens_compressed : {report.tokens_compressed:>8}")
print(f"ratio             : {report.ratio:>8.3f}")
print()
print(f"cost_baseline (USD)   : ${report.cost_baseline:.6f}")
print(f"cost_compressed (USD) : ${report.cost_compressed:.6f}")
print(f"  -> extract cost     : ${report.extras['extract_cost']:.6f}")
print(f"cost_savings_pct      : {report.cost_savings_pct:.2f}%")
print()
print(f"quality_score     : {report.quality_score}/10")
print()
print(f"judge rationale:\n{report.quality_rationale}")


## Honest reading

- If `ratio` is below 0.6 you got a real token reduction.
- If `cost_savings_pct` is positive *and* `quality_score >= 7`, the trade was worth it.
- If `quality_score < 7`, look at `quality_rationale` — usually the judge will tell you which fields the schema dropped. Switch to a richer schema (e.g. `narrative` instead of `qa`) or define a custom one for your domain (see notebook 04).
- On a 1 KB source the extract layer's cost can outweigh the input-token savings. Scale up the input before drawing conclusions.
